# Lab 00 — Environment, Jupyter, and Apple MPS

**Goal:** prove that the training stack is working before we blame the model for anything.

By the end of this lab you should be able to explain:

- what the MPS device is doing;
- where model parameters live;
- why a forward pass is not the same thing as generation;
- approximately how many parameters Qwen3-0.6B has;
- which software versions produced an experiment.

This notebook does **not** train anything.

## 0.1 Reproducibility first

A training run without its environment is an anecdote.

Run this cell and keep the output with your experiment notes.

In [ ]:
import os
import platform
import sys
import torch
import transformers

print("Python:", sys.version.split()[0])
print("macOS/platform:", platform.platform())
print("machine:", platform.machine())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())
print("MPS fallback enabled:", os.getenv("PYTORCH_ENABLE_MPS_FALLBACK"))

### Stop and predict

Before running the next cell: what device do you expect it to print, and what would it mean if it prints `cpu`?

In [ ]:
from tiny_wordle.hardware import preferred_device

device = preferred_device()
device

## 0.2 Load the fixed model

We use **`Qwen/Qwen3-0.6B`** throughout the course.

Do not add quantization yet. Do not add LoRA yet. Do not optimize anything yet.

At this stage, boring is a feature.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)

model = model.to(device)
model.eval()

print(type(model).__name__)
print("model device:", next(model.parameters()).device)
print("model dtype:", next(model.parameters()).dtype)

### Why float32?

We intentionally begin with float32 because precision tricks are a separate variable. Your machine has enough memory that we can first learn the ordinary path, then benchmark lower precision later.

Prediction: roughly how much memory should 600 million float32 parameters require **just for the weights**? Calculate it before running the next cell.

In [ ]:
from tiny_wordle.hardware import trainable_parameter_count

trainable, total = trainable_parameter_count(model)
bytes_for_weights = total * 4

print(f"total parameters:     {total:,}")
print(f"trainable parameters: {trainable:,}")
print(f"approx FP32 weights:  {bytes_for_weights / 1024**3:.2f} GiB")

## 0.3 One forward pass

A causal language model maps token positions to a probability distribution over the vocabulary.

Shape intuition matters. Before running this, predict the shape of `logits`.

In [ ]:
text = "Wordle is a game where"
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

print("input_ids shape:", tuple(inputs["input_ids"].shape))
print("logits shape:", tuple(outputs.logits.shape))
print("vocab size:", tokenizer.vocab_size)

The last dimension is the vocabulary. For every input token position, the model emits one score for every possible next token.

Now inspect the model's top next-token candidates.

In [ ]:
probs = torch.softmax(outputs.logits[0, -1].float(), dim=-1)
top_probs, top_ids = torch.topk(probs, k=10)

for p, token_id in zip(top_probs.tolist(), top_ids.tolist()):
    print(f"{p:8.5f}  {token_id:8d}  {tokenizer.decode([token_id])!r}")

## 0.4 First generation

We will use non-thinking mode for this first sanity check so that generation stays short and observable.

In [ ]:
messages = [
    {"role": "user", "content": "In one sentence, explain the objective of Wordle."}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

model_inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    generated = model.generate(
        **model_inputs,
        max_new_tokens=80,
        do_sample=False,
    )

new_tokens = generated[0, model_inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

## Lab 00 checkpoint

Write down:

1. Your PyTorch and Transformers versions.
2. Whether MPS is available.
3. The actual parameter count.
4. The model's parameter dtype.
5. The difference between `model(**inputs)` and `model.generate(...)`.
6. Anything that surprised you.

Do not move on if MPS is not available. That is an environment problem, not an LLM lesson.